In [1]:
import sys
from importlib.metadata import version
from pathlib import Path

expected_python = Path(r"D:\tutorial\claude-cookbooks\.venv\Scripts\python.exe")
actual_python = Path(sys.executable)
print("当前 Python：", actual_python)
print("anthropic 版本：", version("anthropic"))
print("ipykernel 版本：", version("ipykernel"))

if actual_python.resolve() != expected_python.resolve():
    raise RuntimeError("内核不是 cookbook 项目 Python，请按下一节切换内核。")

print("项目内核检查通过")

当前 Python： D:\tutorial\claude-cookbooks\.venv\Scripts\python.exe
anthropic 版本： 1.12.1
ipykernel 版本： 7.1.0
项目内核检查通过


In [2]:
import os

gateway_url = os.environ.get("ANTHROPIC_BASE_URL", "").rstrip("/")
gateway_key = os.environ.get("ANTHROPIC_API_KEY", "").strip()

print("目标端点：", gateway_url)
print("密钥已配置：", bool(gateway_key))

if gateway_url != "https://qwen.smjtools.com":
    raise RuntimeError("端点不是 cookbook 网关，请检查 Jupyter 的启动命令。")
if not gateway_key or gateway_key == "YOUR_QWEN_GATEWAY_KEY":
    raise RuntimeError("请在 .cookbook.env 中填写你的个人网关密钥。")
if gateway_key.startswith("sk-ant-"):
    raise RuntimeError("检测到 Anthropic 密钥，请不要将它发送到 Qwen 网关。")
if os.environ.get("ANTHROPIC_AUTH_TOKEN"):
    raise RuntimeError("存在额外的 Anthropic auth token，请使用无此变量的独立启动环境。")

print("网关配置检查通过")

目标端点： https://qwen.smjtools.com
密钥已配置： True
网关配置检查通过


In [3]:
from anthropic import Anthropic

with Anthropic(
    api_key=gateway_key,
    base_url=gateway_url,
    timeout=180.0,
    max_retries=0,
) as client:
    with client.messages.stream(
        model="strata",
        max_tokens=128,
        messages=[{"role": "user", "content": "Reply with exactly: pong"}],
    ) as stream:
        print("模型回复：", end="", flush=True)
        for text in stream.text_stream:
            print(text, end="", flush=True)
        result = stream.get_final_message()

print()
print("返回模型：", result.model)
print("停止原因：", result.stop_reason)
print("用量：", result.usage)

if not any(block.type == "text" and block.text.strip() for block in result.content):
    raise RuntimeError("请求结束但没有文本，请检查停止原因和输出 token 上限。")

模型回复：pong
返回模型： qwen3.8-flash-next-iq2_xs
停止原因： end_turn
用量： Usage(cache_creation=None, cache_creation_input_tokens=None, cache_read_input_tokens=0, inference_geo=None, input_tokens=17, output_tokens=2, output_tokens_details=None, server_tool_use=None, service_tier=None)
